# Export the decays of FLIM images based on a preselected ROI

## LIBRARY IMPORTS

In [1]:
# Import all required libraries for FLIM data processing
import tttrlib                   # Library for handling Time-Tagged Time-Resolved (TTTR) data
import pylab as plt              # Matplotlib for plotting
import skimage as ski            # Image processing library
from skimage import io           # For reading/writing images
import numpy as np               # Numerical computations
from mpl_toolkits.axes_grid1 import make_axes_locatable    # For plot formatting
import glob                      # File path pattern matching
import os                        # Operating system interface

##  FILE PATHS AND INSTRUMENT RESPONSE FUNCTION (IRF) SETUP

In [2]:
# Define the folder pattern to process - looks for ROI mask files
path ='C:/Users/Katherina Hemmen/Desktop/ExampleData/Raw_files/Cells/*/*_ROI1.tif'

# Load the Instrument Response Function (IRF) for deconvolution
# IRF represents the system's temporal response and is needed for lifetime analysis
filename_irf = 'C:/Users/Katherina Hemmen/Desktop/ExampleData/Raw_files/Calibration/IRF.ptu'
irf = tttrlib.TTTR(filename_irf)

## CHANNEL AND TIME WINDOW CONFIGURATION

In [3]:
# Define detection channels for PIE (Pulsed Interleaved Excitation) setup
# PIE alternates between two excitation sources to separate donor and acceptor signals
green_ch_s = [0]          # Green channel, perpendicular polarization (s-polarized)
green_ch_p = [3]          # Green channel, parallel polarization (p-polarized)
red_ch_s = [1]            # Red channel, perpendicular polarization
red_ch_p = [2]            # Red channel, parallel polarization

# Time window configuration for PIE separation
# Total time window = 50 ns, TCSPC bin size = 10 ps → 5000 bins total
prompt_range = 0, 2500     # First half: donor excitation period (green laser on)       
delay_range = 2500, 5000   # Second half: acceptor excitation period (red laser on)
dt = 0.02                  # Time resolution per bin in nanoseconds (20 ps) after binning

## IRF PROCESSING FOR EACH CHANNEL

In [4]:
# Extract and process IRF data for each detection channel
# This is used later for deconvolution and background correction

# Get micro times (arrival times within each excitation cycle)
micro_times = irf.micro_times

# Process green channel parallel polarization IRF
green_irf_p_indices = np.array(irf.get_selection_by_channel(green_ch_p))
green_irf_p = micro_times[green_irf_p_indices]
# Bin the data with 2-fold coarsening to match FLIM image binning
green_irf_p_counts = np.bincount(green_irf_p // 2)

# Process green channel perpendicular polarization IRF
green_irf_s_indices = np.array(irf.get_selection_by_channel(green_ch_s))
green_irf_s = micro_times[green_irf_s_indices]
green_irf_s_counts = np.bincount(green_irf_s // 2)

# Process red channel parallel polarization IRF
red_irf_p_indices = np.array(irf.get_selection_by_channel(red_ch_p))
red_irf_p = micro_times[red_irf_p_indices]
red_irf_p_counts = np.bincount(red_irf_p // 2)

# Process red channel perpendicular polarization IRF
red_irf_s_indices = np.array(irf.get_selection_by_channel(red_ch_s))
red_irf_s = micro_times[red_irf_s_indices]
red_irf_s_counts = np.bincount(red_irf_s // 2)

## MAIN PROCESSING LOOP - ITERATE THROUGH ALL FILES

In [9]:
# Process each ROI file found in the specified path
for file in glob.glob(path):
    # Extract base filename without ROI suffix for finding corresponding .ptu file
    filename = os.path.abspath(file).split("_ROI")[0]
    
    # Load the TTTR data file (.ptu format from PicoQuant systems)
    data = tttrlib.TTTR(filename + '.ptu')
    
    # Load the ROI mask image (binary mask defining region of interest)
    mask = io.imread(file).astype(np.uint8)
    
    # Define output filename base for saving results
    save_as = os.path.abspath(file).split(".")[0]
    
    print('Processing....' + filename)
    
    # =============================================================================
    # 1.6. CREATE CLSM IMAGE OBJECTS FOR DIFFERENT CONDITIONS
    # =============================================================================
    # Create CLSM (Confocal Laser Scanning Microscopy) image containers
    # These separate the data by channel and time window
    
    # Green channel images (donor fluorescence)
    clsm_green_p = tttrlib.CLSMImage(tttr_data=data, channels=green_ch_p)
    clsm_green_s = tttrlib.CLSMImage(tttr_data=data, channels=green_ch_s)
    
    # Red channel images during prompt period (FRET-sensitized acceptor)
    clsm_red_prompt_p = tttrlib.CLSMImage(tttr_data=data, channels=red_ch_p, micro_time_ranges=[prompt_range])
    clsm_red_prompt_s = tttrlib.CLSMImage(tttr_data=data, channels=red_ch_s, micro_time_ranges=[prompt_range])
    
    # Red channel images during delay period (directly excited acceptor)
    clsm_red_delay_p = tttrlib.CLSMImage(tttr_data=data, channels=red_ch_p, micro_time_ranges=[delay_range])
    clsm_red_delay_s = tttrlib.CLSMImage(tttr_data=data, channels=red_ch_s, micro_time_ranges=[delay_range])

    # =============================================================================
    # 1.7. FILL IMAGE CONTAINERS WITH DATA
    # =============================================================================
    # Populate each CLSM image container with the appropriate photon data
    clsm_green_p.fill(data, channels=green_ch_p)  # green channels
    clsm_green_s.fill(data, channels=green_ch_s)  # green channels
    clsm_red_prompt_p.fill(data, channels=red_ch_p, micro_time_ranges=[prompt_range])  # red channels, prompt
    clsm_red_prompt_s.fill(data, channels=red_ch_s, micro_time_ranges=[prompt_range])  # red channels, prompt
    clsm_red_delay_s.fill(data, channels=red_ch_s, micro_time_ranges=[delay_range])  # red channels, delay
    clsm_red_delay_p.fill(data, channels=red_ch_p, micro_time_ranges=[delay_range])  # red channels, delay

    # =============================================================================
    # 1.8. GENERATE FLUORESCENCE DECAYS FROM ROI PIXELS
    # =============================================================================
    # Create a 3D mask stack to match the image dimensions
    n_frames, n_lines, n_pixel = clsm_green_p.shape
    stack_mask = np.empty((n_frames, n_lines, n_pixel), dtype=np.uint8)
    for i in range(n_frames):
        stack_mask[i] = mask  # Apply same 2D mask to all frames
    
    # Parameters for decay extraction
    kw_cell = {
        "tttr_data": data,
        "mask": stack_mask,       # ROI mask
        "tac_coarsening": 2,      # 2-fold binning: reduces to 2500 bins (20 ps/bin)
        "stack_frames": True      # Combine all frames
    }
    
    # Extract fluorescence decays for each channel/condition
    # Green channels (donor)
    decay_green_p = clsm_green_p.get_decay_of_pixels(**kw_cell)
    sum_decay_green_p = decay_green_p.sum(axis=0)
    decay_green_s = clsm_green_s.get_decay_of_pixels(**kw_cell)
    sum_decay_green_s = decay_green_s.sum(axis=0)
    
    # Red channels during prompt period (FRET signal)
    decay_red_prompt_p = clsm_red_prompt_p.get_decay_of_pixels(**kw_cell)
    sum_decay_red_prompt_p = decay_red_prompt_p.sum(axis=0)
    decay_red_delay_p = clsm_red_delay_p.get_decay_of_pixels(**kw_cell)
    sum_decay_red_delay_p = decay_red_delay_p.sum(axis=0)
    
    # Red channels during delay period (direct excitation)
    decay_red_prompt_s = clsm_red_prompt_s.get_decay_of_pixels(**kw_cell)
    sum_decay_red_prompt_s = decay_red_prompt_s.sum(axis=0)
    decay_red_delay_s = clsm_red_delay_s.get_decay_of_pixels(**kw_cell)
    sum_decay_red_delay_s = decay_red_delay_s.sum(axis=0)

    # =============================================================================
    # 1.9. GENERATE COMPARISON PLOTS
    # =============================================================================
    # Create figure with three subplots showing different excitation conditions
    fig, ax = plt.subplots(ncols=3, nrows=1, constrained_layout=True, figsize=(9, 3))
    
    # Plot 1: Donor fluorescence (green channels) with IRF for comparison
    ax[0].semilogy(green_irf_p_counts, label='IRF_p', color='gray')        # IRF parallel
    ax[0].semilogy(green_irf_s_counts, label='IRF_s', color='black')       # IRF perpendicular
    ax[0].semilogy(sum_decay_green_p, label='Green_p', color='limegreen')  # Sample parallel
    ax[0].semilogy(sum_decay_green_s, label='Green_s', color='darkgreen')  # Sample perpendicular
    ax[0].legend()
    ax[0].set_title('Donor fluorescence')
    ax[0].set_xlim(0, 1250)   # Show first half of time window (prompt period)
    
    # Plot 2: FRET-sensitized acceptor (red detection during green excitation)
    ax[1].semilogy(red_irf_p_counts, label='IRF_p', color='gray')
    ax[1].semilogy(red_irf_s_counts, label='IRF_s', color='black')
    ax[1].semilogy(sum_decay_red_prompt_p, label='Red_p', color='gold')
    ax[1].semilogy(sum_decay_red_prompt_s, label='Red_s', color='orange')
    ax[1].legend()
    ax[1].set_title('FRET-sensitized acceptor')
    ax[1].set_xlim(0, 1250)   # Prompt period
    
    # Plot 3: Directly excited acceptor (red detection during red excitation)
    ax[2].semilogy(red_irf_p_counts, label='IRF_p', color='gray')
    ax[2].semilogy(red_irf_s_counts, label='IRF_s', color='black')
    ax[2].semilogy(sum_decay_red_delay_p, label='Red_p', color='red')
    ax[2].semilogy(sum_decay_red_delay_s, label='Red_s', color='darkred')
    ax[2].legend()
    ax[2].set_title('Direct excited acceptor')
    ax[2].set_xlim(1250, 2500)    # Delay period
    
    # Save the comparison plot
    plt.savefig(save_as + '_decays.png')
    plt.close()   # Close to save memory
    
    # =============================================================================
    # 1.10. EXPORT DECAY DATA IN JORDI FORMAT
    # =============================================================================
    # Generate time axis for reference
    time_axis = np.arange(0, len(sum_decay_green_p)) * dt

    # Save decay data in JORDI format (used by ChiSurf fitting software)
    # Format: parallel and perpendicular channels concatenated
    
    # Green channel decay (donor, first 1250 bins only)
    jordi_counts_green = np.concatenate([sum_decay_green_p[0:1250], sum_decay_green_s[0:1250]])
    output_green = save_as + '_green.txt'
    np.savetxt(
        output_green, 
        np.vstack([jordi_counts_green]).T
        )
    
    # Red channel prompt decay (FRET-sensitized acceptor)
    jordi_counts_red_prompt = np.concatenate([sum_decay_red_prompt_p[0:1250], sum_decay_red_prompt_s[0:1250]])
    output_red = save_as + '_red_prompt.txt'
    np.savetxt(
        output_red, 
        np.vstack([jordi_counts_red_prompt]).T
        )
    
    # Red channel delay decay (directly excited acceptor, bins 1250-2500 shifted to 0-1250)
    jordi_counts_red_delay = np.concatenate([sum_decay_red_delay_p[1250:2500], sum_decay_red_delay_s[1250:2500]])
    output_red = save_as + '_red_delay.txt'
    np.savetxt(
        output_red, 
        np.vstack([jordi_counts_red_delay]).T
        )

Processing....C:\Users\Katherina Hemmen\Desktop\ExampleData\Raw_files\Cells\cytosolic\eGFP_44
Processing....C:\Users\Katherina Hemmen\Desktop\ExampleData\Raw_files\Cells\DA_1-5\low_FRET_1-5_64
Processing....C:\Users\Katherina Hemmen\Desktop\ExampleData\Raw_files\Cells\DA_1-5\low_FRET_1-5_66
Processing....C:\Users\Katherina Hemmen\Desktop\ExampleData\Raw_files\Cells\DA_1-5\low_FRET_1-5_68
Processing....C:\Users\Katherina Hemmen\Desktop\ExampleData\Raw_files\Cells\DA_1-5\low_FRET_1-5_70
Processing....C:\Users\Katherina Hemmen\Desktop\ExampleData\Raw_files\Cells\DA_1-5\low_FRET_1-5_72
Processing....C:\Users\Katherina Hemmen\Desktop\ExampleData\Raw_files\Cells\DA_1-5\low_FRET_1-5_74
Processing....C:\Users\Katherina Hemmen\Desktop\ExampleData\Raw_files\Cells\DA_1-5\low_FRET_1-5_76
Processing....C:\Users\Katherina Hemmen\Desktop\ExampleData\Raw_files\Cells\Donly\DO_24
Processing....C:\Users\Katherina Hemmen\Desktop\ExampleData\Raw_files\Cells\Donly\high_Donly_32
Processing....C:\Users\Kather

# EXPORT PHOTON STATISTICS AND BACKGROUND ANALYSIS

## SETUP FOR STATISTICS EXPORT

In [10]:
# Define paths for processing all ROI files and output file
path = 'C:/Users/Katherina Hemmen/Desktop/ExampleData/Raw_files/Cells/*/*_ROI.tif'
save_file_as = 'C:/Users/Katherina Hemmen/Desktop/ExampleData/Raw_files/Cells/Photonnumbers.txt'

## INITIALIZE STORAGE LISTS

In [11]:
# Initialize lists to store parameters for all processed files
list_filenames = list()           # Filenames
list_bg_s_green = list()          # Green perpendicular background
list_bg_p_green = list()          # Green parallel background
list_nr_photons_s_green = list()  # Green perpendicular total photons
list_nr_photons_p_green = list()  # Green parallel total photons
list_bg_s_red_prompt = list()     # Red prompt perpendicular background
list_bg_p_red_prompt = list()     # Red prompt parallel background
list_nr_photons_s_red_prompt = list()  # Red prompt perpendicular total photons
list_nr_photons_p_red_prompt = list()  # Red prompt parallel total photons
list_bg_s_red_delay = list()      # Red delay perpendicular background
list_bg_p_red_delay = list()      # Red delay parallel background
list_nr_photons_s_red_delay = list()  # Red delay perpendicular total photons
list_nr_photons_p_red_delay = list()  # Red delay parallel total photons
list_nr_pixel_in_roi = list()     # Number of pixels in each ROI

## PROCESS EACH FILE FOR STATISTICS

In [12]:
for file in glob.glob(path):
    # Construct file paths for the exported decay data
    filename = os.path.abspath(file).split(".")[0]
    data_green = filename  + '_green.txt'            # Green decay file
    data_red_delay = filename + '_red_delay.txt'     # Red delay decay file
    data_red_prompt = filename + '_red_prompt.txt'   # Red prompt decay file
    ROI = file   # ROI mask file
    
    # Load decay data from text files
    decay_green = np.loadtxt(data_green, skiprows=0)
    decay_red_delay = np.loadtxt(data_red_delay, skiprows=0)
    decay_red_prompt = np.loadtxt(data_red_prompt, skiprows=0)
    
    # =============================================================================
    # 2.4. CALCULATE BACKGROUND LEVELS
    # =============================================================================
    # Background is estimated from early time bins (before main fluorescence peak)
    # Using bins 1-75 to avoid bin 0 artifacts and capture baseline noise
    
    # Parallel channel backgrounds
    bg_p_green = np.mean(decay_green[1:75])            # Green parallel background
    bg_p_red_delay = np.mean(decay_red_delay[1:75])    # Red delay parallel background 
    bg_p_red_prompt = np.mean(decay_red_prompt[1:75])  # Red prompt parallel background
    
    # Perpendicular channel backgrounds (offset by 1250 due to data format)
    bg_s_green = np.mean(decay_green[1251:1325])            # Green perpendicular background   
    bg_s_red_delay = np.mean(decay_red_delay[1251:1325])    # Red delay perpendicular background   
    bg_s_red_prompt = np.mean(decay_red_prompt[1251:1325])  # Red prompt perpendicular background

    # =============================================================================
    # 2.5. CALCULATE TOTAL PHOTON COUNTS
    # =============================================================================
    # Sum all photons in each decay curve (includes signal + background)
    
    # Green channel total counts
    sum_p_green = np.sum(decay_green[0:1250])               # Parallel polarization
    sum_s_green = np.sum(decay_green[1250:2500])            # Perpendicular polarization
    
    # Red delay channel total counts
    sum_p_red_delay = np.sum(decay_red_delay[0:1250])       # Parallel
    sum_s_red_delay = np.sum(decay_red_delay[1250:2500])    # Perpendicular
    
    # Red prompt channel total counts 
    sum_p_red_prompt = np.sum(decay_red_prompt[0:1250])     # Parallel
    sum_s_red_prompt = np.sum(decay_red_prompt[1250:2500])  # Perpendicular
    
    # =============================================================================
    # 2.6. COUNT ROI PIXELS
    # =============================================================================
    # Calculate number of pixels in the ROI mask
    ROI_img = io.imread(ROI)
    pixel_in_ROI = np.sum(ROI_img)    # Sum of binary mask = number of True pixels

    # =============================================================================
    # 2.7. STORE RESULTS IN LISTS
    # =============================================================================
    # Append all calculated parameters to their respective lists
    list_filenames.append(str(file))
    list_bg_p_green.append(bg_p_green)
    list_nr_photons_p_green.append(sum_p_green)
    list_bg_s_green.append(bg_s_green)
    list_nr_photons_s_green.append(sum_s_green)
    list_bg_s_red_prompt.append(bg_s_red_prompt)
    list_bg_p_red_prompt.append(bg_p_red_prompt)
    list_nr_photons_s_red_prompt.append(sum_s_red_prompt)
    list_nr_photons_p_red_prompt.append(sum_p_red_prompt)
    list_bg_s_red_delay.append(bg_s_red_delay)
    list_bg_p_red_delay.append(bg_p_red_delay)
    list_nr_photons_s_red_delay.append(sum_s_red_delay)
    list_nr_photons_p_red_delay.append(sum_p_red_delay)
    list_nr_pixel_in_roi.append(pixel_in_ROI)
    
# =============================================================================
# 2.8. EXPORT STATISTICS SUMMARY
# =============================================================================
# Define column headers for the output file
header = 'Filename\tBGgreen(p)\tNrPhotgreen(p)\tBGgreen(s)\tNrPhotgreen(s)\tBGrp(p)\tNrPhotrp(p)\tBGrp(s)\tNrPhotrp(s)\tBGrd(p)\tNrPhotrd(p)\tBGrd(s)\tNrPhotrd(s)\tPixel in ROI'

# Save all statistics as a tab-delimited text file
# Each row represents one ROI file, columns contain different measurements
np.savetxt(
    save_file_as,
    np.vstack(
        [
        list_filenames,                  # File names
        list_bg_p_green,                 # Background levels for all channels
        list_nr_photons_p_green,
        list_bg_s_green,
        list_nr_photons_s_green,
        list_bg_p_red_prompt,
        list_nr_photons_p_red_prompt,
        list_bg_s_red_prompt,
        list_nr_photons_s_red_prompt,
        list_bg_p_red_delay,
        list_nr_photons_p_red_delay,
        list_bg_s_red_delay,
        list_nr_photons_s_red_delay,
        list_nr_pixel_in_roi            # ROI size
        ],
    ).T,
    delimiter='\t', fmt="%s", header=header
)